In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import joblib


# TRAINING PART


file_path = r"E:\END TO END PROJECT\4\predictionData.xlsx"
sheet_name = 'views_churndata'

data = pd.read_excel(file_path, sheet_name=sheet_name)

# Clean column names
data.columns = data.columns.str.strip()

print(data.head())

# Drop unnecessary columns
data = data.drop(['Customer ID', 'Churn Category', 'Churn Reason'], axis=1)

# Encode target
data['Customer Status'] = data['Customer Status'].map({'Stayed': 0, 'Churned': 1})

# Encode categorical columns
categorical_columns = data.select_dtypes(include=['object']).columns
categorical_columns = categorical_columns.drop('Customer Status', errors='ignore')

label_encoders = {}

for column in categorical_columns:
    le = LabelEncoder()
    data[column] = le.fit_transform(data[column].astype(str))
    label_encoders[column] = le

# Split
X = data.drop('Customer Status', axis=1)
y = data['Customer Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate
y_pred = rf_model.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

print("Model trained successfully ")


# PREDICTION PART

sheet_name = 'views_joineddata'

new_data = pd.read_excel(file_path, sheet_name=sheet_name)

# Clean column names
new_data.columns = new_data.columns.str.strip()

print("Columns:", new_data.columns)
print(new_data.head())

# Keep original data
original_data = new_data.copy()

if 'Churn Category' in original_data.columns:
    original_data['Churn Category'] = original_data['Churn Category'].fillna('Others')

if 'Churn Reason' in original_data.columns:
    original_data['Churn Reason'] = original_data['Churn Reason'].fillna('Others')

# Drop only for prediction
columns_to_drop = ['Customer ID', 'Customer Status', 'Churn Category', 'Churn Reason']
new_data = new_data.drop(columns=[col for col in columns_to_drop if col in new_data.columns])

# Encode safely
for column in new_data.select_dtypes(include=['object']).columns:
    
    new_data[column] = new_data[column].fillna('Unknown')
    
    if column in label_encoders:
        le = label_encoders[column]
        
        new_data[column] = new_data[column].apply(
            lambda x: x if x in le.classes_ else 'Unknown'
        )
        
        if 'Unknown' not in le.classes_:
            le.classes_ = np.append(le.classes_, 'Unknown')
        
        new_data[column] = le.transform(new_data[column])

# Predict
new_predictions = rf_model.predict(new_data)

# Add prediction column
original_data['Customer Status Predicted'] = new_predictions

# Filter churned
churned_data = original_data[
    original_data['Customer Status Predicted'] == 1
]

# Save output
output_path = r"E:\END TO END PROJECT\4\CHURNED_FINAL2.csv"
churned_data.to_csv(output_path, index=False)

print("  File saved at:", output_path)

  Customer ID  Gender  Age Married  Number of Dependents          City  \
0  0002-ORFBO  Female   37     Yes                     0  Frazier Park   
1  0003-MKNFE    Male   46      No                     0      Glendale   
2  0004-TLHLJ    Male   50      No                     0    Costa Mesa   
3  0011-IGKFF    Male   78     Yes                     0      Martinez   
4  0013-EXCHZ  Female   75     Yes                     0     Camarillo   

   Zip Code   Latitude   Longitude  Number of Referrals  ...   Payment Method  \
0     93225  34.827662 -118.999073                    2  ...      Credit Card   
1     91206  34.162515 -118.203869                    0  ...      Credit Card   
2     92627  33.645672 -117.922613                    0  ...  Bank Withdrawal   
3     94553  38.014457 -122.115432                    1  ...  Bank Withdrawal   
4     93010  34.227846 -119.079903                    3  ...      Credit Card   

  Monthly Charge Total Charges  Total Refunds Total Extra Data Charg